In [2]:
from kafka import KafkaProducer
p = KafkaProducer(bootstrap_servers="localhost:9092")
print("✅ Connected to Kafka")
p.close()

✅ Connected to Kafka


In [3]:
from kafka.admin import KafkaAdminClient, NewTopic

bootstrap_servers = "localhost:9092"
topic = "sensor_topic"

admin = KafkaAdminClient(bootstrap_servers=bootstrap_servers, client_id="iot-admin")
existing = set(admin.list_topics())
print("Existing topics:", sorted(existing)[:20], "..." if len(existing) > 20 else "")

if topic not in existing:
    admin.create_topics([NewTopic(name=topic, num_partitions=1, replication_factor=1)])
    print(f"✅ Created topic: {topic}")
else:
    print(f"✅ Topic already exists: {topic}")

admin.close()

Existing topics: ['__consumer_offsets', 'ecom.transactions', 'ecom.user_activity', 'iot.temperature', 'iot.vibration', 'sensor_topic'] 
✅ Topic already exists: sensor_topic


In [4]:
import json
from paho.mqtt import client as mqtt

# --- CONFIG (edit to match your working MQTT notebook) ---
MQTT_BROKER = "test.mosquitto.org"
MQTT_PORT = 1883
MQTT_TOPIC = "naven/iot/sensors"   # <-- your Wokwi publish topic

KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC = "sensor_topic"        # <-- the topic you created above

# --- Kafka JSON producer ---
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    acks="all",
    retries=5,
    linger_ms=50,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

print("✅ Kafka producer ready:", KAFKA_BOOTSTRAP, "->", KAFKA_TOPIC)

✅ Kafka producer ready: localhost:9092 -> sensor_topic


In [5]:
def safe_json_loads(s: str):
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

def on_connect(client, userdata, flags, rc, properties=None):
    print("connected:", rc)
    if rc == 0:
        client.subscribe(MQTT_TOPIC)
        print("📡 Subscribed to MQTT topic:", MQTT_TOPIC)
    else:
        print("❌ MQTT connect failed rc=", rc)

def on_message(client, userdata, msg):
    payload = msg.payload.decode("utf-8", errors="replace").strip()
    data = safe_json_loads(payload)

    if not data:
        print("⚠️ Skipping non-JSON payload:", payload)
        return

    # Validate the schema you showed
    required = ("event_time_ms", "temp_c", "vib")
    if not all(k in data for k in required):
        print("⚠️ Missing keys, skipping:", data)
        return

    # Normalize types (helps downstream)
    try:
        data["event_time_ms"] = int(data["event_time_ms"])
        data["temp_c"] = float(data["temp_c"])
        data["vib"] = int(data["vib"])
    except Exception:
        print("⚠️ Bad types, skipping:", data)
        return

    # Send to Kafka
    try:
        md = producer.send(KAFKA_TOPIC, value=data).get(timeout=10)
        print(f"→ Kafka {md.topic}[{md.partition}]@{md.offset}: {data}")
    except Exception as e:
        print("❌ Kafka send failed:", e)

In [ ]:
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

rc = client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
print(rc)  # often prints: MQTTErrorCode.MQTT_ERR_SUCCESS: 0

# Runs forever (stop the cell to stop)
client.loop_forever()

/tmp/ipykernel_1509/2639864217.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


0
connected: 0
📡 Subscribed to MQTT topic: naven/iot/sensors
→ Kafka sensor_topic[0]@918: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032320430, 'temp_c': 34.05, 'vib': 31}
→ Kafka sensor_topic[0]@919: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032321439, 'temp_c': 34.1, 'vib': 25}
→ Kafka sensor_topic[0]@920: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032322442, 'temp_c': 35.21, 'vib': 34}
→ Kafka sensor_topic[0]@921: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032323452, 'temp_c': 35.73, 'vib': 34}
→ Kafka sensor_topic[0]@922: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032324730, 'temp_c': 36.0, 'vib': 29}
→ Kafka sensor_topic[0]@923: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032325739, 'temp_c': 35.47, 'vib': 31}
→ Kafka sensor_topic[0]@924: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 1774032326742, 'temp_c': 36.6, 'vib': 23}
→ Kafka sensor_topic[0]@925: {'device_id': 'esp32-wokwi-001', 'event_time_ms': 177403232